In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os


In [10]:
from openai import OpenAI



openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openrouter.ai/api/v1"#"https://api.groq.com/openai/v1"
    
)

In [3]:
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course'],
    id_field='doc_id',      # <-- use the existing id so if it's re-run, it will update existing records instead of creating duplicates
    db_path='faq.db'
)

In [1]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [2]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [4]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [11]:
response = openai_client.responses.create(
    model='openai/gpt-oss-120b',#"gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseReasoningItem(id='rs_tmp_ma6uvgbzy69', summary=[], type='reasoning', content=[Content(text='The user asks: "I just discovered the course. Can I join it?" Likely they are asking about eligibility to join. The assistant should respond with a brief answer stating they can join if they meet prerequisites, need to enroll via registration etc. Probably no need to call functions. Provide info about enrollment process.', type='reasoning_text')], encrypted_content=None, status='completed', format='unknown'),
 ResponseOutputMessage(id='msg_tmp_mzrun6tdc2c', content=[ResponseOutputText(annotations=[], text='Absolutely—you’re welcome to join the course! Here’s a quick rundown of what you need to do:\n\n1. **Check the prerequisites** – Make sure you meet any required background (e.g., basic programming knowledge, familiarity with the subject area). If the course lists specific prerequisites, you’ll find them on the enrollment page.\n\n2. **Create an account** – If you don’t already have on

In [ ]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [ ]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text